Objetivo do Projeto
Tema:
Análise de Vendas e Otimização da Cadeia de Suprimentos em Supermercados

Objetivo:
Investigar os padrões de compra dos clientes para auxiliar na definição de estratégias de marketing (promoções, segmentação e campanhas) e melhorar a gestão de estoque e logística.

Perguntas de Negócio:

Quais são as linhas de produtos com maior faturamento?

Quais filiais e cidades geram mais vendas?

Como variam as vendas por mês/ano e por dia da semana?

Qual o impacto dos métodos de pagamento nas vendas?

Qual a margem de lucro média por filial?

Há diferença no comportamento entre clientes (Member vs. Normal) e entre gêneros?

Qual é a avaliação média dos clientes para cada linha de produtos?



In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Supermarket_Sales_Project").getOrCreate()

CAMADA BRONZE

In [0]:
# Carregar o CSV para um DataFrame (Camada Bronze)
df_bronze = spark.read.csv("dbfs:/FileStore/supermarket_sales___Sheet1.csv", header=True, inferSchema=True)

# Visualizar os primeiros registros e o schema
df_bronze.show(5)
df_bronze.printSchema()

+-----------+------+---------+-------------+------+--------------------+----------+--------+-------+--------+----------+-------------------+-----------+------+-----------------------+------------+------+
| Invoice ID|Branch|     City|Customer type|Gender|        Product line|Unit price|Quantity| Tax 5%|   Total|      Date|               Time|    Payment|  cogs|gross margin percentage|gross income|Rating|
+-----------+------+---------+-------------+------+--------------------+----------+--------+-------+--------+----------+-------------------+-----------+------+-----------------------+------------+------+
|750-67-8428|     A|   Yangon|       Member|Female|   Health and beauty|     74.69|       7|26.1415|548.9715|2019-01-05|2025-04-07 13:08:00|    Ewallet|522.83|            4.761904762|     26.1415|   9.1|
|226-31-3081|     C|Naypyitaw|       Normal|Female|Electronic access...|     15.28|       5|   3.82|   80.22|2019-03-08|2025-04-07 10:29:00|       Cash|  76.4|            4.761904762| 

In [0]:
# Salvar a camada Bronze em Parquet
df_bronze.write.mode("overwrite").parquet("/dbfs/tmp/bronze_layer/")

CAMADA SILVER

In [0]:
from pyspark.sql.functions import col, to_date, to_timestamp, concat_ws, date_format, month, year
import pyspark.sql.functions as F
##Transformações na Camada Silver

# Inicia as transformações a partir do DataFrame bruto (df_bronze)
    # Converte a coluna "Date" do formato string para um tipo de data (formato "dd/MM/yyyy")
    # Remove espaços em branco da coluna "Time" para evitar problemas de formatação
    # Cria uma nova coluna "Datetime" combinando as colunas "Date" e "Time" 
    # O concat_ws junta as duas colunas com um espaço e converte para timestamp com o formato especificado
    # Extrai o nome do dia da semana (ex.: Monday, Tuesday) a partir da coluna "Date"
    # Extrai o nome do dia da semana (ex.: Monday, Tuesday) a partir da coluna "Date"
    # Extrai o ano da data e cria a coluna "Year"
    # Calcula a margem de lucro percentual: (gross income / Total) * 100
    # Renomeia as colunas para nomes mais simples e consistentes
df_silver = df_bronze \
    .withColumn("Date", to_date(col("Date"), "dd/MM/yyyy")) \
    .withColumn("Time", F.trim(col("Time"))) \
    .withColumn("Datetime", to_timestamp(concat_ws(" ", col("Date"), col("Time")), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("Weekday", date_format(col("Date"), "EEEE")) \
    .withColumn("Month", month(col("Date"))) \
    .withColumn("Year", year(col("Date"))) \
    .withColumn("ProfitMargin", (col("gross income") / col("Total")) * 100) \
    .withColumnRenamed("Invoice ID", "invoice_id") \
    .withColumnRenamed("Customer type", "customer_type") \
    .withColumnRenamed("Product line", "product_line") \
    .withColumnRenamed("Unit price", "unit_price") \
    .withColumnRenamed("Tax 5%", "tax_5") \
    .withColumnRenamed("gross margin percentage", "gross_margin_perc")
    



In [0]:
from pyspark.sql.functions import round

# Arredondando colunas numéricas do df_silver
df_silver_rounded = df_silver \
    .withColumn("unit_price", round(col("unit_price"), 2)) \
    .withColumn("tax_5", round(col("tax_5"), 2)) \
    .withColumn("Total", round(col("Total"), 2)) \
    .withColumn("gross_income", round(col("gross income"), 2)) \
    .withColumn("ProfitMargin", round(col("ProfitMargin"), 2))



In [0]:
# Verificar as transformações
df_silver.show(5)
df_silver.printSchema()

+-----------+------+---------+-------------+------+--------------------+----------+--------+-------+--------+----------+-------------------+-----------+------+-----------------+------------+------+--------+--------+-----+----+-----------------+
| invoice_id|Branch|     City|customer_type|Gender|        product_line|unit_price|Quantity|  tax_5|   Total|      Date|               Time|    Payment|  cogs|gross_margin_perc|gross income|Rating|Datetime| Weekday|Month|Year|     ProfitMargin|
+-----------+------+---------+-------------+------+--------------------+----------+--------+-------+--------+----------+-------------------+-----------+------+-----------------+------------+------+--------+--------+-----+----+-----------------+
|750-67-8428|     A|   Yangon|       Member|Female|   Health and beauty|     74.69|       7|26.1415|548.9715|2019-01-05|2025-04-07 13:08:00|    Ewallet|522.83|      4.761904762|     26.1415|   9.1|    null|Saturday|    1|2019|4.761904761904763|
|226-31-3081|     C|

In [0]:
# Salvar a camada Silver em Parquet
df_silver.write.mode("overwrite").parquet("/dbfs/tmp/silver_layer/")

In [0]:
# Cria uma view SQL chamada 'supermarket_sales_gold' a partir do DataFrame Silver
df_silver.createOrReplaceTempView("supermarket_sales_gold")


CAMADA GOLD

Top 10 Linhas de Produtos por Faturamento:


In [0]:
%sql
SELECT 
    product_line, 
    COUNT(invoice_id) AS num_sales,
    ROUND(SUM(Total), 2) AS total_revenue
FROM supermarket_sales_gold
GROUP BY product_line
ORDER BY total_revenue DESC
LIMIT 10;


product_line,num_sales,total_revenue
Food and beverages,174,56144.84
Sports and travel,166,55122.83
Electronic accessories,170,54337.53
Fashion accessories,178,54305.9
Home and lifestyle,160,53861.91
Health and beauty,152,49193.74


Vendas por Filial e Cidade:


In [0]:
%sql
SELECT 
    Branch,
    City,
    COUNT(invoice_id) AS num_sales,
    ROUND(SUM(Total), 2) AS total_revenue,
    ROUND(AVG(`gross income`), 2) AS avg_gross_income
FROM supermarket_sales_gold
GROUP BY Branch, City
ORDER BY total_revenue DESC;


Branch,City,num_sales,total_revenue,avg_gross_income
C,Naypyitaw,328,110568.71,16.05
A,Yangon,340,106200.37,14.87
B,Mandalay,332,106197.67,15.23


Análise dos Métodos de Pagamento:

In [0]:
%sql
SELECT 
    Payment,
    COUNT(invoice_id) AS num_sales,
    ROUND(SUM(Total), 2) AS total_revenue
FROM supermarket_sales_gold
GROUP BY Payment
ORDER BY total_revenue DESC;


Payment,num_sales,total_revenue
Cash,344,112206.57
Ewallet,345,109993.11
Credit card,311,100767.07


Comportamento por Tipo de Cliente e Gênero:

In [0]:
%sql
SELECT 
    customer_type,
    Gender,
    COUNT(invoice_id) AS num_sales,
    ROUND(SUM(Total), 2) AS total_revenue,
    ROUND(AVG(Rating), 2) AS avg_rating
FROM supermarket_sales_gold
GROUP BY customer_type, Gender
ORDER BY total_revenue DESC;


customer_type,Gender,num_sales,total_revenue,avg_rating
Member,Female,261,88146.94,6.94
Normal,Female,240,79735.98,6.99
Normal,Male,259,79007.32,7.02
Member,Male,240,76076.5,6.94


Vendas Mensais e Tendência Anual:

In [0]:
%sql
SELECT 
    Year,
    Month,
    COUNT(invoice_id) AS num_sales,
    ROUND(SUM(Total), 2) AS total_revenue
FROM supermarket_sales_gold
GROUP BY Year, Month
ORDER BY Year, Month;


Year,Month,num_sales,total_revenue
2019,1,352,116291.87
2019,2,303,97219.37
2019,3,345,109455.51


Análise da Margem de Lucro por Filial:

In [0]:
%sql
SELECT 
    Branch,
    ROUND(AVG(ProfitMargin), 2) AS avg_profit_margin
FROM supermarket_sales_gold
GROUP BY Branch
ORDER BY avg_profit_margin DESC;


Branch,avg_profit_margin
B,4.76
C,4.76
A,4.76


Análise por Dia da Semana:

In [0]:
%sql
SELECT 
    Weekday,
    COUNT(invoice_id) AS num_sales,
    ROUND(SUM(Total), 2) AS total_revenue
FROM supermarket_sales_gold
GROUP BY Weekday
ORDER BY num_sales DESC;


Weekday,num_sales,total_revenue
Saturday,164,56120.81
Tuesday,158,51482.25
Wednesday,143,43731.14
Friday,139,43926.34
Thursday,138,45349.25
Sunday,133,44457.89
Monday,125,37899.08


Avaliação Média por Linha de Produto:

In [0]:
%sql
SELECT 
    product_line,
    ROUND(AVG(Rating), 2) AS avg_rating
FROM supermarket_sales_gold
GROUP BY product_line
ORDER BY avg_rating DESC;


product_line,avg_rating
Food and beverages,7.11
Fashion accessories,7.03
Health and beauty,7.0
Electronic accessories,6.92
Sports and travel,6.92
Home and lifestyle,6.84
